# Notebook 1 — Model untuk Website Sikoshen Store

**Tujuan:** melatih dan mengekspor model yang akan **disambungkan ke website** (backend FastAPI). Notebook ini sengaja dibuat ringkas dan hanya berisi yang dibutuhkan website. Analisis penelitian (EDA, perbandingan 7 algoritma, uji kepekaan, dll.) ada di **Notebook 2**.

| Model | Pertanyaan yang dijawab | Dipakai website untuk |
|---|---|---|
| **Random Forest Regression** (bulanan) | Berapa unit tiap SKU akan terjual **bulan depan**? | Halaman Rekomendasi: prediksi penjualan, stok pengaman, saran jumlah pesan |
| **Random Forest Classifier** (ciri produk) | Kombinasi produk–varian–ukuran seperti apa yang cenderung Fast-Moving? | Perencanaan komposisi belanja stok, pengecekan kombinasi baru dari varian lama |
| **Aturan median** (bukan ML) | SKU ini Fast atau Slow berdasarkan penjualan **yang sudah terjadi**? | Label Fast/Slow di Dashboard dan Produk & Stok |

> **Penting:** label Fast/Slow untuk SKU yang penjualannya sudah tercatat **dihitung langsung dengan aturan median** (Total_Keluar > median jenis produk), karena aturan itu sendiri adalah definisi labelnya. Model klasifikasi tidak dipakai untuk menggantikan aturan ini.

**Keluaran (folder `model_website/`):**

| File | Isi |
|---|---|
| **`rf_sikoshen.pkl`** | **Satu file** (dibuka dengan `joblib.load`) berisi kedua model + daftar fitur + tabel Ramadan + RMSE per SKU + metadata. Taruh di `backend/models/` |
| `sikoshen_pipeline.py` | Langkah pengolahan data + kelas `ModelSikoshen`. Taruh di `backend/` |
| `prediksi_bulan_depan.csv` / `.json` | Hasil prediksi bulan depan siap tampil (opsional, untuk pengecekan) |
| `model_meta.json` | Salinan metadata yang bisa dibaca manusia (opsional) |

**Susunan kunci di dalam `rf_sikoshen.pkl`:** enam kunci pertama **sama persis** dengan `artifact` di notebook penelitian (`clf_model`, `clf_feature_cols`, `reg_model`, `reg_feature_cols`, `ramadan_dates`, `z_layanan`). Ditambah `rmse_per_sku`, `threshold_median`, `sku_master`, `mae`, `meta`, karena stok pengaman dan label Fast/Slow tidak bisa dihitung backend tanpa itu.

## 0. Pengaturan

In [ ]:
import os, sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib, sklearn
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    FOLDER_PATH = '/content/drive/MyDrive/Project Machine Learning/Dataset/'
except ImportError:
    FOLDER_PATH = os.environ.get('SIKOSHEN_DIR', './')      # jalan lokal: atur folder data lewat variabel SIKOSHEN_DIR
if not FOLDER_PATH.endswith('/'):
    FOLDER_PATH += '/'
OUTPUT_DIR = FOLDER_PATH + 'model_website/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

Z_LAYANAN = 1.65          # tingkat layanan ~95% untuk stok pengaman
print('Folder data  :', FOLDER_PATH)
print('Folder output:', OUTPUT_DIR)
print('scikit-learn :', sklearn.__version__, '| pandas:', pd.__version__)

## 1. Tulis modul pipeline

Seluruh langkah pengolahan (stok negatif, tanggal mulai SKU, panel bulanan, fitur lag, kalender Ramadan) ditulis **satu kali** di `sikoshen_pipeline.py`. Modul yang sama dipakai notebook ini saat melatih dan backend saat memprediksi, sehingga fitur saat latih dan saat dipakai **tidak mungkin berbeda**.

In [ ]:
MODUL = r'''"""
sikoshen_pipeline.py
====================
Pipeline + pembungkus model untuk website Sikoshen Store.

Dipakai di DUA tempat supaya hasilnya identik:
  1. Notebook 01_Model_Website_Sikoshen.ipynb (saat melatih model)
  2. Backend website (FastAPI) saat memuat model dan memprediksi

Masukan (data harian dari kartu stok / tabel MySQL) harus berupa DataFrame dengan kolom:
    Jenis_Produk, Varian, Ukuran, Tanggal_dt (datetime), Barang_Masuk, Barang_Keluar, Stok_Akhir
Satu baris = satu SKU pada satu tanggal pencatatan. Ukuran ditulis sebagai teks ("28", "XL").

Model disimpan dalam SATU file (backend/models/rf_sikoshen.pkl), dibuka sekali saat server menyala:

    from pathlib import Path
    from sikoshen_pipeline import muat_model

    MODEL_PATH = Path(__file__).parent / "models" / "rf_sikoshen.pkl"
    model = muat_model(MODEL_PATH)                       # = joblib.load + pembungkus
    hasil = model.prediksi(df_harian)                    # DataFrame prediksi bulan depan per SKU
    p = model.peluang_fast("Tony Jack Biasa", "HITAM", "30")   # peluang Fast-Moving satu kombinasi

Isi dictionary di dalam rf_sikoshen.pkl (hanya tipe dasar + model scikit-learn, tanpa kelas buatan sendiri):
    clf_model, clf_feature_cols   -> Random Forest Classifier (ciri produk) dan urutan fiturnya
    reg_model, reg_feature_cols   -> Random Forest Regression bulanan dan urutan fiturnya
    ramadan_dates, z_layanan      -> tabel Ramadan dan faktor stok pengaman
    rmse_per_sku, threshold_median, sku_master, mae, meta   -> tambahan agar backend bisa menghitung stok pengaman
"""
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

KUNCI_SKU = ['Jenis_Produk', 'Varian', 'Ukuran']
KOLOM_WAJIB = KUNCI_SKU + ['Tanggal_dt', 'Barang_Masuk', 'Barang_Keluar', 'Stok_Akhir']
URUT_HURUF = {'M': 1, 'L': 2, 'XL': 3, 'XXL': 4, 'XXXL': 5, 'XXXXL': 6}
FITUR_KALENDER = ['Porsi_Ramadan', 'Porsi_H14_Lebaran', 'Porsi_Pasca_Lebaran', 'Porsi_Ramadan_lalu']
FITUR_ANGKA = ['lag_1', 'lag_2', 'lag_3', 'rata_3', 'rata_6'] + FITUR_KALENDER + ['Ukuran_Urut']
MIN_CAKUPAN = 0.75       # bulan dengan < 75% hari tercatat tidak dipakai sebagai target
JEDA_MAKS_HARI = 7       # jeda antar-tanggal > 7 hari dianggap periode tidak dicatat
NAMA_BULAN_ID = (
    '', 'Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni',
    'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember',
)

# Awal Ramadan dan hari terakhir Ramadan (sehari sebelum Idul Fitri), versi Pemerintah RI (Kemenag / MABIMS).
# 2023-2026: ketetapan Sidang Isbat Pemerintah RI.
# 2027 (1448 H): perkiraan hisab Kalender Hijriah Indonesia Kemenag RI (MABIMS):
#   1 Ramadan 1448 H = Senin 8 Feb 2027, 1 Syawal 1448 H = Rabu 10 Mar 2027 (hari terakhir Ramadan = 9 Mar 2027).
#   Catatan: status masih perkiraan hisab sebelum Sidang Isbat Kemenag RI (KHGT Muhammadiyah: 1 Syawal 9 Mar 2027).
# WAJIB ditambah setiap tahun (lihat periksa_kalender di bawah).
RAMADAN_DEFAULT = {
    2023: ('2023-03-23', '2023-04-21'),
    2024: ('2024-03-12', '2024-04-09'),
    2025: ('2025-03-01', '2025-03-30'),
    2026: ('2026-02-19', '2026-03-20'),
    2027: ('2027-02-08', '2027-03-09'),
}


# ---------------------------------------------------------------- ukuran & kalender
def urut_ukuran(u):
    """M=1 ... XXXXL=6; celana 28=1 ... 38=11 (urutan ukuran bermakna)."""
    u = str(u).strip()
    return URUT_HURUF[u] if u in URUT_HURUF else int(float(u)) - 27


def porsi_hari(bulan, awal, akhir):
    """Jumlah hari dalam 'bulan' (pd.Period) yang jatuh di rentang [awal, akhir]."""
    s = max(bulan.start_time, pd.Timestamp(awal))
    e = min(bulan.end_time.normalize(), pd.Timestamp(akhir))
    return max(0, (e - s).days + 1)


def _rentang(ramadan):
    return list(ramadan.values()) if isinstance(ramadan, dict) else [tuple(r) for r in ramadan]


def format_bulan_id(bulan):
    """Format pd.Period bulanan menjadi teks Indonesia + YYYY-MM (mis. 'Januari 2027 (2027-01)')."""
    return f'{NAMA_BULAN_ID[bulan.month]} {bulan.year} ({bulan})'


def hitung_kalender(semua_bulan, ramadan):
    """Fitur kalender Ramadan/Lebaran per bulan. Kalender Islam diketahui jauh hari, jadi aman dipakai."""
    rentang = _rentang(ramadan)
    kal = pd.DataFrame({'Bulan': list(semua_bulan)})
    kal['Porsi_Ramadan'] = [sum(porsi_hari(b, a, z) for a, z in rentang) / b.days_in_month for b in semua_bulan]
    kal['Porsi_H14_Lebaran'] = [sum(porsi_hari(b, pd.Timestamp(z) - pd.Timedelta(days=13), z) for a, z in rentang) / 14
                                for b in semua_bulan]
    kal['Porsi_Pasca_Lebaran'] = [sum(porsi_hari(b, pd.Timestamp(z) + pd.Timedelta(days=1),
                                                 pd.Timestamp(z) + pd.Timedelta(days=30)) for a, z in rentang)
                                  / b.days_in_month for b in semua_bulan]
    kal['Porsi_Ramadan_lalu'] = [sum(porsi_hari(b - 1, a, z) for a, z in rentang) / (b - 1).days_in_month
                                 for b in semua_bulan]
    return kal


def bulan_musim_lebaran(kalender):
    """Series bool per bulan: True jika bulan itu Ramadan atau 30 hari setelah Lebaran."""
    k = kalender.set_index('Bulan')
    return (k['Porsi_Ramadan'] + k['Porsi_Pasca_Lebaran']) > 0


def periksa_kalender(ramadan, bulan_prediksi):
    """Hentikan proses jika bulan yang diprediksi melewati tahun terakhir pada tabel Ramadan.
    Kalau tidak dihentikan, fitur kalender akan bernilai 0 secara diam-diam dan prediksi musim Lebaran salah."""
    tahun_terakhir = max(int(pd.Timestamp(z).year) for _, z in _rentang(ramadan))
    if bulan_prediksi.year > tahun_terakhir:
        raise ValueError(
            f'Tabel Ramadan hanya sampai tahun {tahun_terakhir}, padahal bulan yang diprediksi '
            f'{format_bulan_id(bulan_prediksi)}. '
            f'Tambahkan tanggal Ramadan tahun {bulan_prediksi.year} ke RAMADAN lalu latih ulang model.')


# ---------------------------------------------------------------- persiapan data harian
def siapkan_harian(df):
    """Bersihkan data harian: stok negatif -> 0, tentukan tanggal mulai tiap SKU, tambah Ukuran_Urut."""
    kurang = [c for c in KOLOM_WAJIB if c not in df.columns]
    if kurang:
        raise ValueError(f'Kolom wajib tidak ada: {kurang}')
    df = df.copy()
    df['Ukuran'] = df['Ukuran'].astype(str).str.strip()
    df['Tanggal_dt'] = pd.to_datetime(df['Tanggal_dt'])
    df[['Barang_Masuk', 'Barang_Keluar']] = df[['Barang_Masuk', 'Barang_Keluar']].fillna(0).astype(float)
    df = df.sort_values(KUNCI_SKU + ['Tanggal_dt']).reset_index(drop=True)
    df['Stok'] = df['Stok_Akhir'].clip(lower=0)               # stok fisik minimal 0; baris penjualan tetap dipakai
    aktif = (df['Stok_Akhir'] != 0) | (df['Barang_Masuk'] > 0) | (df['Barang_Keluar'] > 0)
    mulai = df[aktif].groupby(KUNCI_SKU)['Tanggal_dt'].min().rename('Tanggal_Mulai').reset_index()
    df = df.merge(mulai, on=KUNCI_SKU, how='left')
    df['Sudah_Dijual'] = df['Tanggal_dt'] >= df['Tanggal_Mulai']   # sebelum SKU ada = "belum dijual", bukan "habis"
    df['Ukuran_Urut'] = df['Ukuran'].map(urut_ukuran)
    return df


def bangun_label_sku(df):
    """Satu baris per SKU + label Fast/Slow (aturan median Total_Keluar per jenis produk)."""
    d = df[df['Sudah_Dijual']]
    sku = d.groupby(KUNCI_SKU).agg(
        Total_Keluar=('Barang_Keluar', 'sum'), Total_Masuk=('Barang_Masuk', 'sum'),
        Tanggal_Mulai=('Tanggal_dt', 'min'), Tanggal_Akhir=('Tanggal_dt', 'max'),
        Rata_Stok=('Stok', 'mean'), Max_Stok=('Stok', 'max'),
        Persen_Hari_Stok_Habis=('Stok', lambda x: (x <= 0).mean() * 100)).reset_index()
    sku['Ukuran_Urut'] = sku['Ukuran'].map(urut_ukuran)
    sku['Threshold_Median'] = sku.groupby('Jenis_Produk')['Total_Keluar'].transform('median')
    sku['Label_enc'] = (sku['Total_Keluar'] > sku['Threshold_Median']).astype(int)
    sku['Label'] = sku['Label_enc'].map({1: 'Fast-Moving', 0: 'Slow-Moving'})
    return sku


# ---------------------------------------------------------------- panel bulanan & fitur regresi
def bangun_panel(df, ramadan):
    """Penjualan per SKU per bulan + kalender + fitur lag. Mengembalikan (panel, kalender, bulan_akhir_data)."""
    bulan_akhir = df['Tanggal_dt'].max().to_period('M')
    semua_bulan = pd.period_range(df['Tanggal_dt'].min().to_period('M'), bulan_akhir + 1, freq='M')
    kalender = hitung_kalender(semua_bulan, ramadan)

    # Cakupan pencatatan per produk per bulan (jeda > JEDA_MAKS_HARI dianggap tidak tercatat)
    cakupan = []
    for produk, g in df.groupby('Jenis_Produk'):
        tgl = pd.Series(sorted(g['Tanggal_dt'].unique()))
        awal, akhir = tgl.min(), tgl.max()
        jeda = [(tgl[i - 1] + pd.Timedelta(days=1), tgl[i] - pd.Timedelta(days=1))
                for i in range(1, len(tgl)) if (tgl[i] - tgl[i - 1]).days > JEDA_MAKS_HARI]
        for b in semua_bulan[:-1]:
            hari = porsi_hari(b, awal, akhir) - sum(porsi_hari(b, a, z) for a, z in jeda)
            cakupan.append({'Jenis_Produk': produk, 'Bulan': b, 'Cakupan': hari / b.days_in_month})
    cakupan = pd.DataFrame(cakupan)

    # Panel SKU x bulan, dimulai dari bulan SKU pertama kali tersedia
    harian = df[df['Sudah_Dijual']].assign(Bulan=lambda t: t['Tanggal_dt'].dt.to_period('M'))
    bulanan = harian.groupby(KUNCI_SKU + ['Bulan'])['Barang_Keluar'].sum().rename('Keluar').reset_index()
    bagian = []
    for kunci, g in bulanan.groupby(KUNCI_SKU):
        penuh = pd.DataFrame({'Bulan': pd.period_range(g['Bulan'].min(), bulan_akhir, freq='M')})
        penuh = penuh.merge(g[['Bulan', 'Keluar']], on='Bulan', how='left')
        for k, v in zip(KUNCI_SKU, kunci):
            penuh[k] = v
        penuh['Keluar'] = penuh['Keluar'].fillna(0).astype(float)
        bagian.append(penuh)
    ukuran = df[KUNCI_SKU + ['Ukuran_Urut']].drop_duplicates()
    panel = (pd.concat(bagian, ignore_index=True)
             .merge(cakupan, on=['Jenis_Produk', 'Bulan'], how='left')
             .merge(kalender, on='Bulan', how='left')
             .merge(ukuran, on=KUNCI_SKU, how='left')
             .sort_values(KUNCI_SKU + ['Bulan']).reset_index(drop=True))
    bulan_pertama = panel.groupby(KUNCI_SKU)['Bulan'].transform('min')
    panel['Valid'] = (panel['Cakupan'] >= MIN_CAKUPAN) & (panel['Bulan'] > bulan_pertama)
    panel['Target'] = pd.to_numeric(panel['Keluar'].where(panel['Valid']), errors='coerce').astype(float)

    # Fitur lag: hanya memakai bulan SEBELUM bulan yang ditebak (float64 agar menampung pecahan & NaN)
    grup = panel.groupby(KUNCI_SKU)['Target']
    for k in [1, 2, 3]:
        panel[f'lag_{k}'] = grup.shift(k).astype(float)
    panel['rata_3'] = pd.to_numeric(
        grup.transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean()), errors='coerce'
    ).astype(float)
    panel['rata_6'] = pd.to_numeric(
        grup.transform(lambda s: s.shift(1).rolling(6, min_periods=2).mean()), errors='coerce'
    ).astype(float)
    for k in [1, 2, 3]:
        panel[f'lag_{k}'] = panel[f'lag_{k}'].fillna(panel['rata_6']).fillna(panel['rata_3'])
    return panel, kalender, bulan_akhir


def matriks_regresi(frame, kolom_acuan=None):
    """Fitur regresi: one-hot Varian + FITUR_ANGKA. Jika kolom_acuan diberikan, kolom disamakan dengan saat latih."""
    X = pd.concat([pd.get_dummies(frame['Varian'], prefix='Varian').astype(int), frame[FITUR_ANGKA]], axis=1)
    if kolom_acuan is not None:
        X = X.reindex(columns=kolom_acuan, fill_value=0)
    return X


def matriks_klasifikasi(frame, kolom_acuan=None):
    """Fitur klasifikasi: one-hot Jenis_Produk + one-hot Varian + Ukuran_Urut (hanya ciri produk)."""
    X = pd.get_dummies(frame[['Jenis_Produk', 'Varian']], prefix=['Produk', 'Varian']).astype(int)
    X['Ukuran_Urut'] = frame['Ukuran_Urut'].values
    if kolom_acuan is not None:
        X = X.reindex(columns=kolom_acuan, fill_value=0)
    return X


def fitur_bulan_depan(panel, kalender, bulan_prediksi):
    """Fitur untuk bulan yang akan diprediksi, dihitung dari penjualan bulan-bulan terakhir tiap SKU."""
    baris = []
    for kunci, g in panel.groupby(KUNCI_SKU):
        g = g.sort_values('Bulan')
        riwayat = g['Target']
        lag = [riwayat.iloc[-k] if len(riwayat) >= k else np.nan for k in [1, 2, 3]]
        rata_3 = riwayat.iloc[-3:].mean()
        rata_6 = riwayat.iloc[-6:].mean() if riwayat.iloc[-6:].notna().sum() >= 2 else np.nan
        lag = [v if pd.notna(v) else (rata_6 if pd.notna(rata_6) else rata_3) for v in lag]
        b = dict(zip(KUNCI_SKU, kunci))
        b.update({'Bulan': bulan_prediksi, 'lag_1': lag[0], 'lag_2': lag[1], 'lag_3': lag[2],
                  'rata_3': rata_3, 'rata_6': rata_6, 'Ukuran_Urut': g['Ukuran_Urut'].iloc[-1]})
        baris.append(b)
    return pd.DataFrame(baris).merge(kalender, on='Bulan', how='left')


def buat_rf_regresi():
    """Setelan Random Forest Regression yang dipakai di penelitian dan di website."""
    from sklearn.ensemble import RandomForestRegressor
    return RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1)


# ---------------------------------------------------------------- pembungkus model untuk backend
def muat_model(path):
    """Buka rf_sikoshen.pkl satu kali (seperti joblib.load(MODEL_PATH) pada contoh Flask) dan bungkus."""
    return ModelSikoshen(path)


class ModelSikoshen:
    """Menjalankan prediksi dari isi rf_sikoshen.pkl. 'sumber' = path file .pkl atau dictionary hasil joblib.load."""

    KUNCI_WAJIB = ['clf_model', 'clf_feature_cols', 'reg_model', 'reg_feature_cols', 'ramadan_dates',
                   'z_layanan', 'rmse_per_sku', 'meta']

    def __init__(self, sumber):
        bundle = joblib.load(sumber) if isinstance(sumber, (str, Path)) else sumber
        kurang = [k for k in self.KUNCI_WAJIB if k not in bundle]
        if kurang:
            raise ValueError(f'File model tidak lengkap, kunci hilang: {kurang}. '
                             f'rmse_per_sku dan meta dibutuhkan untuk stok pengaman; '
                             f'buat ulang rf_sikoshen.pkl dengan notebook 01 versi terbaru.')
        import sklearn
        v_latih = bundle['meta']['versi_library']['scikit_learn']
        if v_latih != sklearn.__version__:
            warnings.warn(f'Model dilatih dengan scikit-learn {v_latih}, server memakai {sklearn.__version__}. '
                          f'Pasang scikit-learn=={v_latih} agar hasil sama.')
        self.bundle = bundle
        self.meta = bundle['meta']
        self.reg = bundle['reg_model']
        self.clf = bundle['clf_model']
        self.rmse = pd.DataFrame(bundle['rmse_per_sku'])
        self.rmse['Ukuran'] = self.rmse['Ukuran'].astype(str)
        self.ramadan = bundle['ramadan_dates']
        self.z = bundle['z_layanan']
        self.fitur_regresi = list(bundle['reg_feature_cols'])
        self.fitur_klasifikasi = list(bundle['clf_feature_cols'])

    def prediksi(self, df_harian, z_layanan=None):
        """Prediksi penjualan bulan depan + stok pengaman + saran pesan per SKU.
        'Bulan depan' = bulan setelah tanggal terakhir pada df_harian."""
        z = self.z if z_layanan is None else z_layanan
        df = siapkan_harian(df_harian)
        bulan_pred = df['Tanggal_dt'].max().to_period('M') + 1
        periksa_kalender(self.ramadan, bulan_pred)
        sku = bangun_label_sku(df)
        panel, kalender, bulan_akhir = bangun_panel(df, self.ramadan)

        depan = fitur_bulan_depan(panel, kalender, bulan_pred)
        X = matriks_regresi(depan, self.fitur_regresi)
        bisa = X.notna().all(axis=1)
        depan['Prediksi_Penjualan'] = np.nan
        if bisa.any():
            depan.loc[bisa, 'Prediksi_Penjualan'] = np.maximum(0, self.reg.predict(X[bisa]))

        musim = 'lebaran' if bool(bulan_musim_lebaran(kalender)[bulan_pred]) else 'biasa'
        kol = f'RMSE_{musim}'
        depan = depan.merge(self.rmse[KUNCI_SKU + [kol]].rename(columns={kol: 'RMSE_SKU'}), on=KUNCI_SKU, how='left')
        depan['RMSE_SKU'] = depan['RMSE_SKU'].fillna(self.rmse[kol].median())
        stok_kini = (df.sort_values('Tanggal_dt').groupby(KUNCI_SKU)['Stok'].last()
                     .rename('Stok_Saat_Ini').reset_index())
        depan = depan.merge(stok_kini, on=KUNCI_SKU, how='left').merge(
            sku[KUNCI_SKU + ['Label', 'Threshold_Median', 'Total_Keluar']], on=KUNCI_SKU, how='left')
        depan['Stok_Pengaman'] = np.ceil(z * depan['RMSE_SKU'])
        depan['Saran_Pesan'] = np.ceil(np.maximum(0, depan['Prediksi_Penjualan'] + depan['Stok_Pengaman']
                                                  - depan['Stok_Saat_Ini']))
        depan['Prediksi_Penjualan'] = depan['Prediksi_Penjualan'].round(0)
        depan['Musim'] = musim
        depan['Bulan'] = depan['Bulan'].astype(str)
        return depan[KUNCI_SKU + ['Bulan', 'Musim', 'Label', 'Prediksi_Penjualan', 'Stok_Saat_Ini',
                                  'Stok_Pengaman', 'Saran_Pesan', 'RMSE_SKU']]

    def peluang_fast(self, jenis_produk, varian, ukuran):
        """Peluang (0-1) kombinasi produk-varian-ukuran cenderung Fast-Moving, dari ciri produk saja.
        Hanya valid untuk jenis produk dan varian yang sudah pernah dijual (varian baru ~ tebakan acak)."""
        baris = pd.DataFrame([{'Jenis_Produk': jenis_produk, 'Varian': varian, 'Ukuran_Urut': urut_ukuran(ukuran)}])
        if f'Varian_{varian}' not in self.fitur_klasifikasi:
            warnings.warn(f'Varian "{varian}" belum pernah ada saat model dilatih; hasil tidak dapat diandalkan.')
        X = matriks_klasifikasi(baris, self.fitur_klasifikasi)
        return float(self.clf.predict_proba(X)[0, 1])
'''

Path(OUTPUT_DIR + 'sikoshen_pipeline.py').write_text(MODUL, encoding='utf-8')
if OUTPUT_DIR not in sys.path:
    sys.path.insert(0, OUTPUT_DIR)
import importlib, sikoshen_pipeline as sik
importlib.reload(sik)
KUNCI_SKU = sik.KUNCI_SKU
print('Modul ditulis ke', OUTPUT_DIR + 'sikoshen_pipeline.py')

## 2. Muat data kartu stok

Jika 3 file Excel ada di folder data, data **dibangun ulang dari Excel** (aturan penggabungan sama dengan notebook penelitian). Jika tidak, dipakai `dataset_gabungan_sikoshen_diperbaiki.csv`.

Di website, langkah ini digantikan oleh data dari MySQL yang formatnya sama (kolom wajib: `Jenis_Produk, Varian, Ukuran, Tanggal_dt, Barang_Masuk, Barang_Keluar, Stok_Akhir`).

In [ ]:
import re, openpyxl

FILE_EXCEL = {
    'Iron Head Ultimate': dict(file='iron head ultimate.xlsx', sheet='Sheet2', baris_header=2, kol_varian=1, kol_ukuran=2, baris_data=5),
    'Kemeja Denim':       dict(file='STOK KMJ DENIM-40.xlsx',   sheet='Sheet1', baris_header=2, kol_varian=1, kol_ukuran=3, baris_data=5),
    'Tony Jack Biasa':    dict(file='STOK TJ BIASA-50.xlsx',    sheet='Sheet1', baris_header=2, kol_varian=2, kol_ukuran=4, baris_data=4),
}

def cari_file(kata_kunci, bawaan):
    # Cari file .xlsx di folder data yang namanya memuat semua kata kunci (abaikan huruf besar/kecil, spasi, _ , -, dan akhiran seperti _1).
    for f in sorted(os.listdir(FOLDER_PATH)):
        nama = re.sub(r'[^a-z0-9]+', ' ', f.lower())
        if f.lower().endswith('.xlsx') and not f.startswith('~$') and all(k in nama for k in kata_kunci):
            return f
    return bawaan

for _p, _k in {'Iron Head Ultimate': ['iron', 'head'], 'Kemeja Denim': ['kmj', 'denim'], 'Tony Jack Biasa': ['tj', 'biasa']}.items():
    FILE_EXCEL[_p]['file'] = cari_file(_k, FILE_EXCEL[_p]['file'])
    print(f"{_p:20s}-> {FILE_EXCEL[_p]['file']}")

def baca_kartu_stok(produk, cfg):
    # Mengubah satu file Excel kartu stok (format lebar) menjadi data harian (format panjang).
    wb = openpyxl.load_workbook(FOLDER_PATH + cfg['file'], data_only=True, read_only=True)
    rows = list(wb[cfg['sheet']].iter_rows(values_only=True))
    header = rows[cfg['baris_header'] - 1]
    kolom = []
    for i, v in enumerate(header):
        if not isinstance(v, str):
            continue
        t = v.strip().lower()
        if t.startswith('jumlah'):
            tgl = re.search(r'(\d+)/(\d+)/(\d+)', t).groups()
            kolom.append((i, 'J', f'{int(tgl[0])}/{int(tgl[1])}/{tgl[2][-2:]}'))   # '923' -> '23' (perbaikan salah ketik)
        elif 'masuk' in t:
            kolom.append((i, 'M', None))
        elif 'keluar' in t:
            kolom.append((i, 'K', None))
    hasil, varian = [], None
    for r in rows[cfg['baris_data'] - 1:]:
        if r is None or all(x is None for x in r):
            continue
        if r[cfg['kol_varian']] is not None:
            varian = str(r[cfg['kol_varian']]).strip()
        ukuran = r[cfg['kol_ukuran']]
        if ukuran is None:
            continue
        if isinstance(ukuran, float) and ukuran.is_integer():
            ukuran = int(ukuran)
        ukuran = str(ukuran).strip()
        masuk = keluar = 0.0
        stok_kemarin = None
        for i, jenis, tgl in kolom:
            v = r[i] if i < len(r) else None
            if jenis == 'M':
                masuk += v or 0
            elif jenis == 'K':
                keluar += v or 0
            else:
                if v is None:
                    continue
                if stok_kemarin is None:
                    saldo_awal, m, k, opname = v, 0.0, 0.0, 0.0
                else:
                    saldo_awal, m, k = 0.0, masuk, keluar
                    opname = v - (stok_kemarin + masuk - keluar)
                hasil.append(dict(Jenis_Produk=produk, Varian=varian, Ukuran=ukuran, Tanggal=tgl,
                                  Saldo_Awal=saldo_awal, Barang_Masuk=m, Barang_Keluar=k,
                                  Penyesuaian_Opname=opname, Stok_Akhir=v))
                stok_kemarin, masuk, keluar = v, 0.0, 0.0
    return pd.DataFrame(hasil)

if all(os.path.exists(FOLDER_PATH + c['file']) for c in FILE_EXCEL.values()):
    h = pd.concat([baca_kartu_stok(p, c) for p, c in FILE_EXCEL.items()], ignore_index=True)
    h['Tanggal_dt'] = pd.to_datetime(h['Tanggal'], format='%d/%m/%y')
    df_mentah = (h.groupby(KUNCI_SKU + ['Tanggal_dt'], sort=False)
                  .agg(Barang_Masuk=('Barang_Masuk', 'sum'), Barang_Keluar=('Barang_Keluar', 'sum'),
                       Penyesuaian_Opname=('Penyesuaian_Opname', 'sum'), Saldo_Awal=('Saldo_Awal', 'sum'),
                       Stok_Akhir=('Stok_Akhir', 'last')).reset_index())
    sumber = 'file Excel'
else:
    df_mentah = pd.read_csv(FOLDER_PATH + 'dataset_gabungan_sikoshen_diperbaiki.csv', dtype={'Ukuran': str})
    df_mentah['Tanggal_dt'] = pd.to_datetime(df_mentah['Tanggal'], format='%d/%m/%y')
    sumber = 'dataset_gabungan_sikoshen_diperbaiki.csv'

df_mentah = df_mentah.sort_values(KUNCI_SKU + ['Tanggal_dt']).reset_index(drop=True)
print(f'Sumber data : {sumber}')
print(f'Baris harian: {len(df_mentah):,} | SKU: {df_mentah.groupby(KUNCI_SKU).ngroups}')
print(f"Periode     : {df_mentah['Tanggal_dt'].min().date()} s.d. {df_mentah['Tanggal_dt'].max().date()}")

# Validasi: stok hari ini = stok kemarin + saldo awal + masuk - keluar + opname (harus 100%)
kemarin = df_mentah.groupby(KUNCI_SKU)['Stok_Akhir'].shift().fillna(0)
cocok = np.isclose(df_mentah['Stok_Akhir'], kemarin + df_mentah['Saldo_Awal'] + df_mentah['Barang_Masuk']
                   - df_mentah['Barang_Keluar'] + df_mentah['Penyesuaian_Opname'])
print(f'Baris memenuhi rumus stok: {cocok.mean()*100:.2f}%')
assert cocok.mean() > 0.999, 'Rumus stok tidak cocok: periksa penggabungan data.'

## 3. Persiapan data, label Fast/Slow, dan thresholdnya

Label Fast/Slow = aturan median per jenis produk. **Nilai median ini disimpan** supaya website memakai ambang yang sama.

In [ ]:
df = sik.siapkan_harian(df_mentah)
df_sku = sik.bangun_label_sku(df)

print(f'Jumlah SKU unik: {len(df_sku)}')
print('\nThreshold median Total_Keluar per jenis produk:')
print(df_sku.groupby('Jenis_Produk')['Threshold_Median'].first().to_string())
print('\nDistribusi label:')
print(pd.crosstab(df_sku['Jenis_Produk'], df_sku['Label'], margins=True, margins_name='Total'))

## 4. Panel bulanan dan fitur regresi

Data harian diringkas menjadi **penjualan per SKU per bulan**. Bulan dengan pencatatan < 75% (mis. pemilik berhenti mencatat saat Lebaran) **tidak dipakai sebagai target**, karena nol di bulan itu berarti "tidak dicatat", bukan "tidak laku".

In [ ]:
RAMADAN = sik.RAMADAN_DEFAULT
BULAN_AKHIR_DATA = df['Tanggal_dt'].max().to_period('M')
BULAN_PREDIKSI = BULAN_AKHIR_DATA + 1
sik.periksa_kalender(RAMADAN, BULAN_PREDIKSI)
panel, kalender, BULAN_AKHIR_DATA = sik.bangun_panel(df, RAMADAN)

Xr = sik.matriks_regresi(panel)
siap = panel['Target'].notna() & Xr.notna().all(axis=1)
print(f'Panel SKU x bulan   : {len(panel):,} baris')
print(f"Target valid        : {panel['Valid'].sum():,} baris | siap latih: {siap.sum():,} | jumlah fitur: {Xr.shape[1]}")
print(f'Data terakhir       : {BULAN_AKHIR_DATA}  ->  bulan yang diprediksi: {BULAN_PREDIKSI}')
tdk = panel.loc[~panel['Valid'] & (panel['Bulan'] > panel.groupby(KUNCI_SKU)['Bulan'].transform('min'))]
print('\nBulan tidak dipakai sebagai target (pencatatan tidak lengkap):')
print(tdk.groupby(['Jenis_Produk', 'Bulan']).size().rename('Jumlah_SKU').to_string() or '-')

## 5. Evaluasi bergulir 12 bulan (hanya Random Forest vs tebakan sederhana)

Untuk setiap bulan dari 12 bulan terakhir, model dilatih ulang **hanya dengan data sebelum bulan itu**, lalu menebak bulan tersebut. Hasilnya dipakai untuk dua hal: (1) angka akurasi yang ditampilkan di website, (2) **kesalahan per SKU** sebagai dasar stok pengaman.

Perbandingan lengkap dengan algoritma lain ada di Notebook 2.

In [ ]:
bulan_uji = pd.period_range(BULAN_AKHIR_DATA - 11, BULAN_AKHIR_DATA, freq='M')
hasil_uji = []
for T in bulan_uji:
    latih = siap & (panel['Bulan'] < T)
    uji = siap & (panel['Bulan'] == T)
    if uji.sum() == 0 or latih.sum() < 30:
        continue
    catat = panel.loc[uji, KUNCI_SKU + ['Bulan', 'Target']].copy()
    catat['Baseline bulan lalu'] = panel.loc[uji, 'lag_1']
    catat['Baseline rata 6 bln'] = panel.loc[uji, 'rata_6']
    m = sik.buat_rf_regresi().fit(Xr[latih], panel.loc[latih, 'Target'])
    catat['Random Forest'] = np.maximum(0, m.predict(Xr[uji]))
    hasil_uji.append(catat)
hasil_uji = pd.concat(hasil_uji, ignore_index=True)

def metrik_reg(aktual, tebak):
    e = tebak - aktual
    return {'MAE': np.abs(e).mean(), 'RMSE': np.sqrt((e ** 2).mean()), 'WAPE': np.abs(e).sum() / aktual.sum()}

tabel_reg = pd.DataFrame({n: metrik_reg(hasil_uji['Target'].values, hasil_uji[n].values)
                          for n in ['Random Forest', 'Baseline bulan lalu', 'Baseline rata 6 bln']}).T
print(f"Bulan uji: {hasil_uji['Bulan'].min()} s.d. {hasil_uji['Bulan'].max()} | baris uji: {len(hasil_uji):,}")
tabel_reg.round(3)

## 6. Model klasifikasi (ciri produk)

Fitur **hanya ciri produk** (jenis produk, varian, ukuran), bukan data penjualan/stok, agar tidak terjadi *target leakage* (lihat bagian 7 Notebook 2). Setelan memakai hasil tuning di penelitian: `n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=1`.

**Model akhir dilatih pada seluruh 126 SKU** (bukan 80% seperti saat evaluasi di notebook penelitian). Pembagian 80/20 hanya untuk mengukur akurasi; setelah setelan terbaik ditemukan, model produksi dilatih ulang dengan semua data agar tidak membuang 26 SKU.

Peluang Fast yang ditampilkan memakai **prediksi out-of-fold** (SKU ditebak oleh model yang tidak pernah melihat SKU tersebut), sehingga angkanya jujur.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

Xc = sik.matriks_klasifikasi(df_sku)
yc = df_sku['Label_enc']
PARAM_KLAS = dict(n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=1, random_state=42)

peluang_oof = np.zeros(len(Xc))
for tr, te in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(Xc, yc):
    m = RandomForestClassifier(**PARAM_KLAS).fit(Xc.iloc[tr], yc.iloc[tr])
    peluang_oof[te] = m.predict_proba(Xc.iloc[te])[:, 1]
akurasi_oof = accuracy_score(yc, (peluang_oof > 0.5).astype(int))

clf_final = RandomForestClassifier(**PARAM_KLAS).fit(Xc, yc)      # model akhir: semua SKU
df_sku['Peluang_Fast'] = peluang_oof
print(f'Akurasi out-of-fold klasifikasi: {akurasi_oof*100:.1f}%  (jumlah SKU: {len(Xc)})')

## 7. Model regresi final dan kesalahan per SKU

Model akhir dilatih dengan **semua** data bulanan yang valid. Kesalahan (RMSE) per SKU dihitung dari evaluasi bergulir, dipisah menjadi **musim biasa** dan **musim Lebaran** (Ramadan + 30 hari sesudahnya), karena kesalahan di musim Lebaran jauh lebih besar. SKU tanpa data pada salah satu musim memakai median musim tersebut.

In [ ]:
reg_final = sik.buat_rf_regresi().fit(Xr[siap], panel.loc[siap, 'Target'])

lebaran = sik.bulan_musim_lebaran(kalender)
hasil_uji['Musim'] = np.where(hasil_uji['Bulan'].map(lebaran), 'lebaran', 'biasa')
rmse = (hasil_uji.assign(e2=lambda t: (t['Random Forest'] - t['Target']) ** 2)
        .groupby(KUNCI_SKU + ['Musim'])['e2'].mean().pow(0.5).unstack('Musim').reset_index())
for musim, lain in [('biasa', 'lebaran'), ('lebaran', 'biasa')]:
    if musim not in rmse.columns:
        rmse[musim] = np.nan
    rmse[musim] = rmse[musim].fillna(rmse[musim].median())
    rmse[musim] = rmse[musim].fillna(rmse[lain].median() if lain in rmse.columns else 0)   # musim tidak pernah teruji
rmse = rmse.rename(columns={'biasa': 'RMSE_biasa', 'lebaran': 'RMSE_lebaran'})[KUNCI_SKU + ['RMSE_biasa', 'RMSE_lebaran']]
print(f'SKU dengan RMSE: {len(rmse)} dari {len(df_sku)}')
rmse.describe().round(2)

## 8. Prediksi bulan depan (di dalam notebook)

In [ ]:
depan = sik.fitur_bulan_depan(panel, kalender, BULAN_PREDIKSI)
X_depan = sik.matriks_regresi(depan, list(Xr.columns))
bisa = X_depan.notna().all(axis=1)
depan['Prediksi_Penjualan'] = np.nan
depan.loc[bisa, 'Prediksi_Penjualan'] = np.maximum(0, reg_final.predict(X_depan[bisa]))
print(f'Bulan diprediksi: {BULAN_PREDIKSI} | SKU yang bisa diprediksi: {bisa.sum()} dari {len(depan)}')
depan[KUNCI_SKU + ['lag_1', 'rata_3', 'rata_6', 'Prediksi_Penjualan']].head(8).round(1)

## 9. Ekspor ke satu file `rf_sikoshen.pkl`

Semua yang dibutuhkan backend dibungkus dalam **satu dictionary**, lalu disimpan dengan `joblib.dump`, sama seperti `artifact` di notebook penelitian. Isinya hanya model scikit-learn dan tipe dasar Python (list, dict, angka, teks), tanpa kelas buatan sendiri, sehingga file bisa dibuka tanpa meng-import apa pun selain scikit-learn.

In [ ]:
sku_master = df_sku[KUNCI_SKU + ['Ukuran_Urut', 'Total_Keluar', 'Threshold_Median', 'Label', 'Peluang_Fast']].copy()

meta = {
    'versi_model': pd.Timestamp.now().strftime('%Y%m%d-%H%M'),
    'dilatih_pada': str(pd.Timestamp.now()),
    'sumber_data': sumber,
    'periode_data': [str(df['Tanggal_dt'].min().date()), str(df['Tanggal_dt'].max().date())],
    'bulan_data_terakhir': str(BULAN_AKHIR_DATA),
    'bulan_prediksi': str(BULAN_PREDIKSI),
    'jumlah_sku': int(len(df_sku)),
    'jumlah_baris_latih_regresi': int(siap.sum()),
    'aturan_label': "Fast-Moving jika Total_Keluar SKU > median Total_Keluar jenis produknya; selain itu Slow-Moving. "
                    "Dihitung langsung dari data (tanpa ML) untuk SKU yang penjualannya sudah tercatat.",
    'param_klasifikasi': PARAM_KLAS,
    'evaluasi_regresi_bergulir': {
        'bulan_uji': [str(hasil_uji['Bulan'].min()), str(hasil_uji['Bulan'].max())],
        **{n: {k: round(float(v), 4) for k, v in tabel_reg.loc[n].items()} for n in tabel_reg.index}},
    'evaluasi_klasifikasi': {'akurasi_out_of_fold': round(float(akurasi_oof), 4),
                             'catatan': 'Tidak berlaku untuk varian yang belum pernah dijual (akurasi ~50%).'},
    'kalender_ramadan': {
        'cakupan_tahun': [min(RAMADAN), max(RAMADAN)],
        'sumber_2023_2026': 'Ketetapan Sidang Isbat Pemerintah RI (Kemenag RI).',
        'sumber_2027': 'Perkiraan hisab Kalender Hijriah Indonesia Kemenag RI (kriteria MABIMS): 1 Ramadan 1448 H = 8 Feb 2027, 1 Syawal 1448 H = 10 Mar 2027 (akhir Ramadan = 9 Mar 2027).',
        'status_2027': 'perkiraan_hisab_menunggu_sidang_isbat',
    },
    'versi_library': {'scikit_learn': sklearn.__version__, 'pandas': pd.__version__, 'numpy': np.__version__,
                      'joblib': joblib.__version__},
}

artifact = {
    # ---- Enam kunci ini SAMA PERSIS dengan notebook penelitian ----
    'clf_model': clf_final,                          # Model A: klasifikasi Fast/Slow (dilatih pada 126 SKU)
    'clf_feature_cols': list(Xc.columns),
    'reg_model': reg_final,                          # Model B: regresi penjualan bulan depan
    'reg_feature_cols': list(Xr.columns),
    'ramadan_dates': RAMADAN,
    'z_layanan': Z_LAYANAN,
    # ---- Tambahan yang dibutuhkan backend ----
    'rmse_per_sku': rmse.to_dict('records'),         # dasar stok pengaman (musim biasa dan Lebaran)
    'threshold_median': {k: float(v) for k, v in df_sku.groupby('Jenis_Produk')['Threshold_Median'].first().items()},
    'sku_master': sku_master.to_dict('records'),
    'mae': float(tabel_reg.loc['Random Forest', 'MAE']),
    'meta': meta,
}
MODEL_FILE = OUTPUT_DIR + 'rf_sikoshen.pkl'
joblib.dump(artifact, MODEL_FILE, compress=3)
Path(OUTPUT_DIR + 'model_meta.json').write_text(json.dumps({**meta, 'ramadan': {str(k): list(v) for k, v in RAMADAN.items()}},
                                                           indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Tersimpan: {MODEL_FILE}  ({os.path.getsize(MODEL_FILE)/1e6:.1f} MB)')
print('Kunci di dalam file:', list(artifact))

## 10. Uji sambung: buka `rf_sikoshen.pkl` lalu prediksi dari data harian

Ini meniru yang dilakukan backend: **memuat file model + memberi data harian → mendapat prediksi**. Hasilnya harus **sama persis** dengan prediksi di dalam notebook (bagian 8). Jika tidak sama, ada yang salah pada ekspor.

In [ ]:
import importlib
importlib.reload(sik)
bundle = joblib.load(MODEL_FILE)                   # persis cara backend membuka file model
print('Kunci hasil joblib.load:', list(bundle))
model = sik.ModelSikoshen(bundle)                   # atau sik.muat_model(MODEL_FILE)
hasil = model.prediksi(df_mentah)                  # data harian mentah -> prediksi (seperti dari MySQL)

# 1) Prediksi dari model yang dimuat ulang harus sama dengan model di memori
cek = hasil.merge(depan[KUNCI_SKU + ['Prediksi_Penjualan']], on=KUNCI_SKU, suffixes=('', '_nb'))
selisih = (cek['Prediksi_Penjualan'].round(0) - cek['Prediksi_Penjualan_nb'].round(0)).abs().max()
print(f'Selisih maksimum prediksi (model dimuat vs notebook): {selisih}')
assert selisih == 0

# 2) Label dari aturan median harus sama dengan label di sku_master
lab = hasil.merge(df_sku[KUNCI_SKU + ['Label']], on=KUNCI_SKU, suffixes=('', '_ref'))
assert (lab['Label'] == lab['Label_ref']).all()

# 3) Kelengkapan
assert len(hasil) == len(df_sku) and hasil['Prediksi_Penjualan'].notna().all() and hasil['Saran_Pesan'].notna().all()
print(f'OK: {len(hasil)} SKU, bulan {hasil["Bulan"].iloc[0]}, musim {hasil["Musim"].iloc[0]}')

# 4) Peluang Fast satu kombinasi
contoh = df_sku.iloc[0]
print('Contoh peluang_fast:', contoh['Jenis_Produk'], contoh['Varian'], contoh['Ukuran'], '->',
      round(model.peluang_fast(contoh['Jenis_Produk'], contoh['Varian'], contoh['Ukuran']), 3))

hasil.to_csv(OUTPUT_DIR + 'prediksi_bulan_depan.csv', index=False, encoding='utf-8-sig')
hasil.to_json(OUTPUT_DIR + 'prediksi_bulan_depan.json', orient='records', force_ascii=False, indent=2)
print('\nTotal per jenis produk:')
display(hasil.groupby('Jenis_Produk')[['Prediksi_Penjualan', 'Stok_Saat_Ini', 'Saran_Pesan']].sum())
print('10 SKU dengan saran pesan terbanyak:')
display(hasil.sort_values('Saran_Pesan', ascending=False).head(10))

## 11. Cara menyambungkan ke backend

Susunan folder (mengikuti struktur `sikoshen-store` Anda):

```
backend/
├── models/
│   └── rf_sikoshen.pkl          <- hasil notebook ini (satu file)
├── sikoshen_pipeline.py         <- hasil notebook ini
├── main.py  db.py  requirements.txt ...
```

1. Salin `rf_sikoshen.pkl` ke `backend/models/` dan `sikoshen_pipeline.py` ke `backend/`.
2. Di `requirements.txt` kunci versi scikit-learn yang tercatat di `model_meta.json` → `versi_library` (file `.pkl` dari scikit-learn versi lain bisa gagal dibuka atau berubah hasilnya).
3. Buka file **sekali** saat server menyala (sama seperti contoh di slide), lalu pakai di setiap request:

```python
from pathlib import Path
import joblib
from sikoshen_pipeline import ModelSikoshen

MODEL_PATH = Path(__file__).parent / "models" / "rf_sikoshen.pkl"
bundle = joblib.load(MODEL_PATH)          # dictionary: clf_model, reg_model, feature_cols, ...
model = ModelSikoshen(bundle)             # pembungkus: prediksi, stok pengaman, saran pesan

def ambil_kartu_stok_dari_mysql():
    # DataFrame dengan kolom: Jenis_Produk, Varian, Ukuran (teks), Tanggal_dt (datetime),
    #                         Barang_Masuk, Barang_Keluar, Stok_Akhir
    ...

# FastAPI
@app.post("/api/predictions/next-month")
def prediksi_bulan_depan():
    return model.prediksi(ambil_kartu_stok_dari_mysql()).to_dict("records")

# Flask (jika backend memakai Flask)
@app.post("/api/predictions/next-month")
def prediksi_bulan_depan():
    return jsonify(model.prediksi(ambil_kartu_stok_dari_mysql()).to_dict("records"))
```

**Catatan operasional**
* **Latih ulang** notebook ini setiap data baru masuk (minimal tiap bulan), lalu timpa `rf_sikoshen.pkl`.
* Bulan yang diprediksi selalu **bulan setelah tanggal terakhir di data**. Jika data berhenti Mei 2026, hasilnya untuk Juni 2026, bukan "bulan ini".
* Tabel `RAMADAN` **harus ditambah tiap tahun**. Jika bulan prediksi melewati tahun terakhir pada tabel, program sengaja berhenti dengan pesan error (bukan menebak salah).
* Prediksi adalah **penjualan** (Barang_Keluar), bukan stok. Saran pesan = Prediksi + Stok pengaman − Stok saat ini (minimal 0), dan belum memperhitungkan waktu tunggu pemasok atau modal.
* Label Fast/Slow di website dihitung dengan aturan median (kolom `Label` pada hasil `prediksi()`), bukan oleh model.
* File `.pkl` hanya boleh dibuka dari sumber yang dipercaya (milik tim sendiri), karena membuka file pickle dapat menjalankan kode.